# Contract-first Clinical NLP — Kaggle Run All

Notebook này chỉ điều phối API runtime. Business logic nằm trong
`clinical_nlp_lab`; không train local và không copy logic lớn vào notebook.

Trên Kaggle, notebook mặc định clone branch `codex/kaggle-end-to-end-pipeline`
từ GitHub (Internet phải bật), chỉ cần attach Dataset dữ liệu theo
`KAGGLE_RUNBOOK.md`, bật GPU, rồi chạy `Save Version → Run All`. Có thể đặt
`GIT_CLONE_URL`, `GIT_CLONE_REF` hoặc `PROJECT_ROOT_OVERRIDE` trong setup cell.
Kaggle Run All là bước nghiệm thu do người dùng thực hiện.


In [ ]:
from __future__ import annotations

import json
import os
import importlib.util
import subprocess
import sys
from dataclasses import replace
from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").is_dir()
PROJECT_ROOT_OVERRIDE = os.environ.get("PROJECT_ROOT_OVERRIDE", "")
GIT_CLONE_URL = os.environ.get("GIT_CLONE_URL", "https://github.com/takumi612/AI-Race-Viettel.git")
GIT_CLONE_REF = os.environ.get("GIT_CLONE_REF", "codex/kaggle-end-to-end-pipeline")
GIT_CLONE_DIR = Path(os.environ.get("GIT_CLONE_DIR", "/kaggle/working/AI-Race-Viettel"))
USE_GIT_CLONE = os.environ.get("USE_GIT_CLONE", "1" if IS_KAGGLE else "0") == "1"
RUN_MODE = os.environ.get("RUN_MODE", "full")
RUN_ID = os.environ.get("RUN_ID", "") or None
ENABLE_QWEN_RERANKER = False
QWEN_GPU_MEMORY_UTILIZATION = 0.50

def log_step(step: int, status: str, message: str, **context):
    marker = {"START": "STEP_START", "END": "STEP_END", "ERROR": "STEP_ERROR"}.get(status, "STEP_INFO")
    payload = {"message": message, **context}
    print(f"[{marker}] STEP {step} {json.dumps(payload, ensure_ascii=False, default=str)}", flush=True)

EXPECTED_STEP_LABELS = ("STEP 1", "STEP 2", "STEP 3", "STEP 4", "STEP 5", "STEP 6", "STEP 7", "STEP 8", "STEP 9")

KAGGLE_OPTIONS = {
    "enable_qwen_reranker": ENABLE_QWEN_RERANKER,
    "qwen_gpu_memory_utilization": QWEN_GPU_MEMORY_UTILIZATION,
}
# Compatibility contract for the legacy pipeline adapter:
# run_inference(enable_qwen_reranker=ENABLE_QWEN_RERANKER,
#               qwen_gpu_memory_utilization=QWEN_GPU_MEMORY_UTILIZATION)

PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser() if PROJECT_ROOT_OVERRIDE.strip() else Path.cwd()
if IS_KAGGLE and not PROJECT_ROOT_OVERRIDE.strip() and USE_GIT_CLONE:
    clone_project_root = GIT_CLONE_DIR / "v2"
    if not (clone_project_root / "clinical_nlp_lab").is_dir():
        if GIT_CLONE_DIR.exists() and any(GIT_CLONE_DIR.iterdir()):
            raise FileExistsError(
                f"GIT_CLONE_DIR exists but is not the expected repository: {GIT_CLONE_DIR}"
            )
        print(f"[GIT_CLONE] cloning {GIT_CLONE_URL}@{GIT_CLONE_REF} -> {GIT_CLONE_DIR}", flush=True)
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", GIT_CLONE_REF, GIT_CLONE_URL, str(GIT_CLONE_DIR)],
            check=True,
        )
    PROJECT_ROOT = clone_project_root
if not (PROJECT_ROOT / "clinical_nlp_lab").is_dir():
    candidates = [path.parent for path in Path("/kaggle/input").rglob("clinical_nlp_lab") if path.is_dir()] if IS_KAGGLE else []
    if candidates:
        PROJECT_ROOT = candidates[0]
if not (PROJECT_ROOT / "clinical_nlp_lab").is_dir():
    raise FileNotFoundError(
        "clinical_nlp_lab is unavailable; enable Internet for the default git clone "
        "or set PROJECT_ROOT_OVERRIDE/USE_GIT_CLONE=0"
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if os.environ.get("INSTALL_RUNTIME_DEPS", "1") == "1":
    requirements = PROJECT_ROOT / "requirements-kaggle.txt"
    required_modules = ("transformers", "accelerate", "sentencepiece", "safetensors")
    missing_modules = [module for module in required_modules if importlib.util.find_spec(module) is None]
    if missing_modules and requirements.is_file():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)

from clinical_nlp_lab.kaggle_phases import build_kaggle_phase_runners
from clinical_nlp_lab.orchestration import (
    PHASES,
    LatestPointer,
    RunConfig,
    execute_run,
    finish_run,
    resume_run,
    run_inference_only,
    run_phase,
    start_run,
)

DATASET_ROOT = Path(os.environ.get("DATASET_ROOT", ""))
if not str(DATASET_ROOT):
    dataset_candidates = list(Path("/kaggle/input").rglob("synthetic_train_v2")) if IS_KAGGLE else []
    DATASET_ROOT = dataset_candidates[0] if dataset_candidates else Path("../data_v2/Training_data/synthetic_train_v2")
ARTIFACT_DIR = Path(os.environ.get("ARTIFACT_DIR", str(Path("/kaggle/working/artifacts") if IS_KAGGLE else PROJECT_ROOT / "artifacts")))
ARTIFACT_SOURCE_DIR = Path(os.environ.get("ARTIFACT_SOURCE_DIR", str(PROJECT_ROOT / "artifacts")))
INPUT_SOURCE = Path(os.environ.get("INPUT_SOURCE", "input.zip"))
if not INPUT_SOURCE.exists():
    input_candidates = [
        PROJECT_ROOT / "input",
        DATASET_ROOT.parent / "input",
        *(list(Path("/kaggle/input").rglob("input.zip")) if IS_KAGGLE else []),
        *(list(Path("/kaggle/input").rglob("input")) if IS_KAGGLE else []),
    ]
    INPUT_SOURCE = next((candidate for candidate in input_candidates if candidate.exists()), INPUT_SOURCE)
MODEL_SOURCE = os.environ.get("MODEL_SOURCE", "xlm-roberta-base")
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", "/kaggle/working/run_output" if IS_KAGGLE else "artifacts/run_output"))
CONFIG_PATH = Path(os.environ.get("CONFIG_PATH", str(ARTIFACT_DIR / "config.json")))
EXPECTED_GPU_COUNT = int(os.environ.get("EXPECTED_GPU_COUNT", "2"))

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()

# On Kaggle the writable artifact directory is intentionally empty at startup;
# phase 01 copies the read-only artifact bundle into it.  Fingerprint the source
# config in that case so resume/contract validation is bound before phase 01.
CONFIG_SOURCE_PATH = ARTIFACT_SOURCE_DIR / "config.json"
CONFIG_FINGERPRINT_PATH = CONFIG_PATH if CONFIG_PATH.is_file() else CONFIG_SOURCE_PATH
CONFIG_FINGERPRINT = sha256_file(CONFIG_FINGERPRINT_PATH) if CONFIG_FINGERPRINT_PATH.is_file() else "unbound"
DATASET_FINGERPRINT = "unbound"
provenance_path = DATASET_ROOT / "reports" / "dataset_provenance.json"
if provenance_path.is_file():
    provenance_payload = json.loads(provenance_path.read_text(encoding="utf-8"))
    DATASET_FINGERPRINT = str(provenance_payload.get("dataset", {}).get("fingerprint", "unbound"))

config = RunConfig(
    run_mode=RUN_MODE,
    run_id=RUN_ID,
    dataset_root=DATASET_ROOT,
    output_dir=OUTPUT_DIR,
    artifact_dir=ARTIFACT_DIR,
    artifact_source_dir=ARTIFACT_SOURCE_DIR,
    input_source=INPUT_SOURCE,
    model_source=MODEL_SOURCE,
    config_path=CONFIG_PATH,
    expected_gpu_count=EXPECTED_GPU_COUNT,
    use_distributed=os.environ.get("USE_DISTRIBUTED", "1") == "1",
    fast_dev_run=os.environ.get("FAST_DEV_RUN", "0") == "1",
    dataset_fingerprint=DATASET_FINGERPRINT,
    config_fingerprint=CONFIG_FINGERPRINT,
)
config = replace(config, phase_runners=build_kaggle_phase_runners(config))

if RUN_MODE == "full":
    ACTIVE_PHASES = PHASES
    SESSION = start_run(config, phases=ACTIVE_PHASES)
elif RUN_MODE == "resume":
    latest_path = Path(config.output_dir) / "LATEST.json"
    if not latest_path.is_file():
        raise FileNotFoundError(f"Missing resume pointer: {latest_path}")
    latest = LatestPointer(**json.loads(latest_path.read_text(encoding="utf-8")))
    start_index = PHASES.index(latest.phase) + 1
    ACTIVE_PHASES = PHASES[start_index:]
    SESSION = start_run(config, phases=ACTIVE_PHASES, run_id=latest.run_id)
elif RUN_MODE == "inference_only":
    ACTIVE_PHASES = (PHASES[0], PHASES[1], PHASES[2], PHASES[11], PHASES[12])
    SESSION = start_run(config, phases=ACTIVE_PHASES)
else:
    raise ValueError(f"Unsupported RUN_MODE: {RUN_MODE!r}")

log_step(1, "END", "Runtime session opened", run_mode=RUN_MODE, active_phases=list(ACTIVE_PHASES), run_id=SESSION.run_id)
# Batch APIs remain available for non-notebook callers: execute_run(config), resume_run(config, latest), run_inference_only(config, bundle).


## Phase 01: Preflight dữ liệu và provenance

`phase_01_preflight` chạy qua runner trong `clinical_nlp_lab.kaggle_phases`; artifact và log được publish sau khi phase kết thúc.

In [ ]:
PHASE_NAME = "phase_01_preflight"
PHASE_INDEX = 1
if PHASE_NAME in ACTIVE_PHASES:
    log_step(PHASE_INDEX, "START", "Starting phase", phase=PHASE_NAME)
    PHASE_RESULT = run_phase(SESSION, PHASE_NAME)
    log_step(PHASE_INDEX, "END", "Phase completed", phase=PHASE_NAME, result=PHASE_RESULT)
    print(json.dumps(PHASE_RESULT, ensure_ascii=False, indent=2, default=str))
else:
    print(json.dumps({"phase": PHASE_NAME, "status": "SKIPPED", "run_mode": RUN_MODE}, ensure_ascii=False))


## Phase 02: Resolve nguồn input/model/KB

`phase_02_resolve_sources` chạy qua runner trong `clinical_nlp_lab.kaggle_phases`; artifact và log được publish sau khi phase kết thúc.

In [ ]:
PHASE_NAME = "phase_02_resolve_sources"
PHASE_INDEX = 2
if PHASE_NAME in ACTIVE_PHASES:
    log_step(PHASE_INDEX, "START", "Starting phase", phase=PHASE_NAME)
    PHASE_RESULT = run_phase(SESSION, PHASE_NAME)
    log_step(PHASE_INDEX, "END", "Phase completed", phase=PHASE_NAME, result=PHASE_RESULT)
    print(json.dumps(PHASE_RESULT, ensure_ascii=False, indent=2, default=str))
else:
    print(json.dumps({"phase": PHASE_NAME, "status": "SKIPPED", "run_mode": RUN_MODE}, ensure_ascii=False))


## Phase 03: Inventory model và resource budget

`phase_03_inventory_models` chạy qua runner trong `clinical_nlp_lab.kaggle_phases`; artifact và log được publish sau khi phase kết thúc.

In [ ]:
PHASE_NAME = "phase_03_inventory_models"
PHASE_INDEX = 3
if PHASE_NAME in ACTIVE_PHASES:
    log_step(PHASE_INDEX, "START", "Starting phase", phase=PHASE_NAME)
    PHASE_RESULT = run_phase(SESSION, PHASE_NAME)
    log_step(PHASE_INDEX, "END", "Phase completed", phase=PHASE_NAME, result=PHASE_RESULT)
    print(json.dumps(PHASE_RESULT, ensure_ascii=False, indent=2, default=str))
else:
    print(json.dumps({"phase": PHASE_NAME, "status": "SKIPPED", "run_mode": RUN_MODE}, ensure_ascii=False))


## Phase 04: Build record metadata

`phase_04_build_metadata` chạy qua runner trong `clinical_nlp_lab.kaggle_phases`; artifact và log được publish sau khi phase kết thúc.

In [ ]:
PHASE_NAME = "phase_04_build_metadata"
PHASE_INDEX = 4
if PHASE_NAME in ACTIVE_PHASES:
    log_step(PHASE_INDEX, "START", "Starting phase", phase=PHASE_NAME)
    PHASE_RESULT = run_phase(SESSION, PHASE_NAME)
    log_step(PHASE_INDEX, "END", "Phase completed", phase=PHASE_NAME, result=PHASE_RESULT)
    print(json.dumps(PHASE_RESULT, ensure_ascii=False, indent=2, default=str))
else:
    print(json.dumps({"phase": PHASE_NAME, "status": "SKIPPED", "run_mode": RUN_MODE}, ensure_ascii=False))


## Phase 05: Build fixed/OOF splits

`phase_05_build_splits` chạy qua runner trong `clinical_nlp_lab.kaggle_phases`; artifact và log được publish sau khi phase kết thúc.

In [ ]:
PHASE_NAME = "phase_05_build_splits"
PHASE_INDEX = 5
if PHASE_NAME in ACTIVE_PHASES:
    log_step(PHASE_INDEX, "START", "Starting phase", phase=PHASE_NAME)
    PHASE_RESULT = run_phase(SESSION, PHASE_NAME)
    log_step(PHASE_INDEX, "END", "Phase completed", phase=PHASE_NAME, result=PHASE_RESULT)
    print(json.dumps(PHASE_RESULT, ensure_ascii=False, indent=2, default=str))
else:
    print(json.dumps({"phase": PHASE_NAME, "status": "SKIPPED", "run_mode": RUN_MODE}, ensure_ascii=False))


## Phase 06: Prepare owner-window training contract

`phase_06_prepare_training_contract` chạy qua runner trong `clinical_nlp_lab.kaggle_phases`; artifact và log được publish sau khi phase kết thúc.

In [ ]:
PHASE_NAME = "phase_06_prepare_training_contract"
PHASE_INDEX = 6
if PHASE_NAME in ACTIVE_PHASES:
    log_step(PHASE_INDEX, "START", "Starting phase", phase=PHASE_NAME)
    PHASE_RESULT = run_phase(SESSION, PHASE_NAME)
    log_step(PHASE_INDEX, "END", "Phase completed", phase=PHASE_NAME, result=PHASE_RESULT)
    print(json.dumps(PHASE_RESULT, ensure_ascii=False, indent=2, default=str))
else:
    print(json.dumps({"phase": PHASE_NAME, "status": "SKIPPED", "run_mode": RUN_MODE}, ensure_ascii=False))


## Phase 07: Curriculum Stage 1

`phase_07_stage1` chạy qua runner trong `clinical_nlp_lab.kaggle_phases`; artifact và log được publish sau khi phase kết thúc.

In [ ]:
PHASE_NAME = "phase_07_stage1"
PHASE_INDEX = 7
if PHASE_NAME in ACTIVE_PHASES:
    log_step(PHASE_INDEX, "START", "Starting phase", phase=PHASE_NAME)
    PHASE_RESULT = run_phase(SESSION, PHASE_NAME)
    log_step(PHASE_INDEX, "END", "Phase completed", phase=PHASE_NAME, result=PHASE_RESULT)
    print(json.dumps(PHASE_RESULT, ensure_ascii=False, indent=2, default=str))
else:
    print(json.dumps({"phase": PHASE_NAME, "status": "SKIPPED", "run_mode": RUN_MODE}, ensure_ascii=False))


## Phase 08: Curriculum Stage 2

`phase_08_stage2` chạy qua runner trong `clinical_nlp_lab.kaggle_phases`; artifact và log được publish sau khi phase kết thúc.

In [ ]:
PHASE_NAME = "phase_08_stage2"
PHASE_INDEX = 8
if PHASE_NAME in ACTIVE_PHASES:
    log_step(PHASE_INDEX, "START", "Starting phase", phase=PHASE_NAME)
    PHASE_RESULT = run_phase(SESSION, PHASE_NAME)
    log_step(PHASE_INDEX, "END", "Phase completed", phase=PHASE_NAME, result=PHASE_RESULT)
    print(json.dumps(PHASE_RESULT, ensure_ascii=False, indent=2, default=str))
else:
    print(json.dumps({"phase": PHASE_NAME, "status": "SKIPPED", "run_mode": RUN_MODE}, ensure_ascii=False))


## Phase 09: Curriculum Stage 3

`phase_09_stage3` chạy qua runner trong `clinical_nlp_lab.kaggle_phases`; artifact và log được publish sau khi phase kết thúc.

In [ ]:
PHASE_NAME = "phase_09_stage3"
PHASE_INDEX = 9
if PHASE_NAME in ACTIVE_PHASES:
    log_step(PHASE_INDEX, "START", "Starting phase", phase=PHASE_NAME)
    PHASE_RESULT = run_phase(SESSION, PHASE_NAME)
    log_step(PHASE_INDEX, "END", "Phase completed", phase=PHASE_NAME, result=PHASE_RESULT)
    print(json.dumps(PHASE_RESULT, ensure_ascii=False, indent=2, default=str))
else:
    print(json.dumps({"phase": PHASE_NAME, "status": "SKIPPED", "run_mode": RUN_MODE}, ensure_ascii=False))


## Phase 10: Final-fit encoder

`phase_10_final_fit` chạy qua runner trong `clinical_nlp_lab.kaggle_phases`; artifact và log được publish sau khi phase kết thúc.

In [ ]:
PHASE_NAME = "phase_10_final_fit"
PHASE_INDEX = 10
if PHASE_NAME in ACTIVE_PHASES:
    log_step(PHASE_INDEX, "START", "Starting phase", phase=PHASE_NAME)
    PHASE_RESULT = run_phase(SESSION, PHASE_NAME)
    log_step(PHASE_INDEX, "END", "Phase completed", phase=PHASE_NAME, result=PHASE_RESULT)
    print(json.dumps(PHASE_RESULT, ensure_ascii=False, indent=2, default=str))
else:
    print(json.dumps({"phase": PHASE_NAME, "status": "SKIPPED", "run_mode": RUN_MODE}, ensure_ascii=False))


## Phase 11: Fit assertion/candidate heads

`phase_11_fit_heads` chạy qua runner trong `clinical_nlp_lab.kaggle_phases`; artifact và log được publish sau khi phase kết thúc.

In [ ]:
PHASE_NAME = "phase_11_fit_heads"
PHASE_INDEX = 11
if PHASE_NAME in ACTIVE_PHASES:
    log_step(PHASE_INDEX, "START", "Starting phase", phase=PHASE_NAME)
    PHASE_RESULT = run_phase(SESSION, PHASE_NAME)
    log_step(PHASE_INDEX, "END", "Phase completed", phase=PHASE_NAME, result=PHASE_RESULT)
    print(json.dumps(PHASE_RESULT, ensure_ascii=False, indent=2, default=str))
else:
    print(json.dumps({"phase": PHASE_NAME, "status": "SKIPPED", "run_mode": RUN_MODE}, ensure_ascii=False))


## Phase 12: Inference raw-offset + KB recovery

`phase_12_inference` chạy qua runner trong `clinical_nlp_lab.kaggle_phases`; artifact và log được publish sau khi phase kết thúc.

In [ ]:
PHASE_NAME = "phase_12_inference"
PHASE_INDEX = 12
if PHASE_NAME in ACTIVE_PHASES:
    log_step(PHASE_INDEX, "START", "Starting phase", phase=PHASE_NAME)
    PHASE_RESULT = run_phase(SESSION, PHASE_NAME)
    log_step(PHASE_INDEX, "END", "Phase completed", phase=PHASE_NAME, result=PHASE_RESULT)
    print(json.dumps(PHASE_RESULT, ensure_ascii=False, indent=2, default=str))
else:
    print(json.dumps({"phase": PHASE_NAME, "status": "SKIPPED", "run_mode": RUN_MODE}, ensure_ascii=False))


## Phase 13: Validate và package artifacts

`phase_13_packaging` chạy qua runner trong `clinical_nlp_lab.kaggle_phases`; artifact và log được publish sau khi phase kết thúc.

In [ ]:
PHASE_NAME = "phase_13_packaging"
PHASE_INDEX = 13
if PHASE_NAME in ACTIVE_PHASES:
    log_step(PHASE_INDEX, "START", "Starting phase", phase=PHASE_NAME)
    PHASE_RESULT = run_phase(SESSION, PHASE_NAME)
    log_step(PHASE_INDEX, "END", "Phase completed", phase=PHASE_NAME, result=PHASE_RESULT)
    print(json.dumps(PHASE_RESULT, ensure_ascii=False, indent=2, default=str))
else:
    print(json.dumps({"phase": PHASE_NAME, "status": "SKIPPED", "run_mode": RUN_MODE}, ensure_ascii=False))


In [ ]:
if SESSION.completed and SESSION.completed[-1] == ACTIVE_PHASES[-1]:
    SUMMARY = finish_run(SESSION)
    log_step(9, "END", "Run completed", status=SUMMARY.status, phase=SUMMARY.phase_completed)
    print(json.dumps(SUMMARY.__dict__, ensure_ascii=False, default=str))
else:
    print(json.dumps({"status": "INCOMPLETE", "completed": SESSION.completed, "next": ACTIVE_PHASES[len(SESSION.completed):]}, ensure_ascii=False))
